# 🏛️ FinRL-X-MT5: The K-Dense Council Quickstart

**Institutional-Grade Mixture-of-Experts (MoE) Trading System for MetaTrader 5**
- **Target Assets**: Indices & Commodities (`NAS100.x`, `WTI.x`, `XAGUSD.x`, `US30.x`, `SPX500.x`, `GER40.x`)
- **Backtest Mode**: MT5 Strategy Tester — *Every tick based on real ticks*
- **Edge**: MT5 High-Frequency Ticks $\times$ Yahoo/FMP EOD Cross-Asset Fundamental Fusion
- **Architecture**: 5-Expert Council gated by **NSGA-III Pareto Optimization**

---

## 1. Connect to MT5 & Inspect Account State

In [ ]:
import MetaTrader5 as mt5
from src.data.mt5_tick_fetcher import MT5TickFetcher
from src.trading.mt5_manager import MT5Manager
from src.config.settings import settings

manager = MT5Manager()
snap = manager.get_account_snapshot()

print(f"Broker:   {snap.server}")
print(f"Account:  {snap.login}")
print(f"Equity:   ${snap.equity:,.2f}")
print(f"Balance:  ${snap.balance:,.2f}")
print(f"Margin:   ${snap.margin_free:,.2f} free (Level: {snap.margin_level:.1f}%)")
print(f"Leverage: 1:{snap.leverage}")

## 2. Inspect Target Symbols & Broker Specifications

In [ ]:
symbols = ["NAS100.x", "WTI.x", "XAGUSD.x", "US30.x", "SPX500.x", "GER40.x", "JAP225.x", "UK100.x", "AUS200.x"]

for s in symbols:
    spec = manager.get_symbol_spec(s)
    if spec:
        print(f"{s:<10} | Contract: {spec['contract_size']:>6} | Spread: {spec['spread']:>4} pts | Lot: [{spec['min_lot']}-{spec['max_lot']}, step {spec['lot_step']}]")

## 3. Pull Real M5 Bars & Microstructure Features

In [ ]:
from src.data.tick_feature_engineer import TickFeatureEngineer

symbol = "NAS100.x"
fetcher = MT5TickFetcher()
raw_bars = fetcher.get_ohlcv(symbol, timeframe="M5", n_bars=300)

engineer = TickFeatureEngineer(timeframe_minutes=5)
bars_feat = engineer.compute_features(raw_bars)

print(f"Generated {bars_feat.shape[1]} technical & microstructure features over {len(bars_feat)} bars")
print("Sample features:", bars_feat.columns[:15])

## 4. Cross-Asset Correlation Fusion (The Edge Layer)
Fuses MT5 intraday bars with Yahoo Finance ETF & constituent flow signals (`QQQ`, `AAPL`, `MSFT`, `NVDA`, etc.).

In [ ]:
from src.data.yahoo_fetcher import YahooFetcher
from src.data.correlation_fuser import CorrelationFuser

yahoo = YahooFetcher()
yahoo_df = yahoo.get_correlation_features(symbol, days=45)

fuser = CorrelationFuser()
fused_df = fuser.fuse(bars_feat, yahoo_df, symbol=symbol)

print(f"Unified Feature Matrix shape: {fused_df.shape}")
fused_df.tail(3)

## 5. Convene the K-Dense Council & NSGA-III Gate
The 5 experts evaluate the market state in sequence:
1. **Expert 2 (HMM)** identifies regime (`BULL`, `BEAR`, `SIDEWAYS`)
2. **Expert 1 (DRL)** computes policy distribution
3. **Expert 3 (TimesFM)** forecasts trajectory
4. **Expert 4 (SHAP Analyst)** scores fundamental edge attribution
5. **Expert 5 (PyMC Actuary)** sets Bayesian TP/SL
6. **NSGA-III Gate** blends all expert signals using Pareto-optimal weights.

In [ ]:
from src.council.council import KDenseCouncil

council = KDenseCouncil()
decision = council.evaluate(symbol, fused_df)

print("=" * 60)
print(f"🏛️ COUNCIL DECISION FOR {symbol}")
print(f"Direction:        {'LONG' if decision.direction > 0 else 'SHORT' if decision.direction < 0 else 'FLAT'}")
print(f"Consensus Signal: {decision.consensus_signal:.4f}")
print(f"Confidence:       {decision.council_confidence:.1%}")
print(f"Market Regime:    {decision.regime}")
print(f"Take Profit:      {decision.tp_price}")
print(f"Stop Loss:        {decision.sl_price}")
print(f"Expected RR:      {decision.expected_rr:.2f}")
print(f"Tradeable:        {'✅ YES' if decision.is_tradeable else '❌ NO'}")
print("=" * 60)

## 6. Export Signals to MT5 Strategy Tester
Generates `finrl_x_signals.csv` and deploys it automatically to your MT5 terminal directory so that `FinRL_X_MT5.mq5` can be backtested under **Every tick based on real ticks**.

In [ ]:
from src.backtest.mt5_tick_backtest import MT5TickBacktestBridge

bridge = MT5TickBacktestBridge(council=council)
csv_path = bridge.generate_strategy_tester_signals(symbol=symbol, days=30)
print(f"Signals deployed to: {csv_path}")

## 7. Python Fast Simulation Backtest & Risk Analytics

In [ ]:
from src.backtest.backtest_engine import BacktestEngine
from src.trading.performance_analyzer import PerformanceAnalyzer

engine = BacktestEngine(initial_balance=10_000.0)
metrics, equity_df, trades = engine.run(symbol, fused_df.to_pandas(), council)

print(PerformanceAnalyzer.format_report(metrics, f"{symbol} Simulation Performance"))